# DCGAN

Convolutional generator and discriminator for the same 100-d prior as the vanilla GAN. Batch norm, LeakyReLU, and the strided 7→14→28 layout are the architectural changes Q3 asks you to compare.

Run this on a Colab **GPU** runtime. The project folder must sit at `MyDrive/dl-sesi07`. Change `DRIVE_ROOT` in the setup cell if your folder name differs. Run the cells from top to bottom. Results are written to `artifacts/` on Drive.


In [ ]:
import os
import sys
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/dl-sesi07"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path(DRIVE_ROOT)
else:
    ROOT = Path.cwd().resolve()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if "google.colab" in sys.modules:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])


## Setup

The prior is projected to `128 × 7 × 7`, then two stride-2 transposed convolutions reach 14 and 28. The discriminator is the stride-2 mirror. Weights start at `N(0, 0.02)`.


In [ ]:
%matplotlib inline
import time

import torch
from torch.optim import Adam
from tqdm.auto import tqdm

from src.config import (
    ALPHAS,
    FID_SAMPLES,
    GRID_SAMPLES,
    NOISE_DIM,
    SEED,
    get_device,
    set_seed,
    standard_normal,
    to_signed,
    to_unit,
)
from src.data import fixed_test_images, get_splits, make_loader
from src.io import load_checkpoint, load_json, save_json, save_shared, save_torch
from src.metrics import gan_stability, measure_latency_ms
from src.models import DCGANGenerator as Generator, DCGANDiscriminator as Discriminator, dcgan_weights
from src.plots import show_grid

set_seed(SEED)
DEVICE = get_device()
EPOCHS = 50
NAME = "dcgan"
print("device", DEVICE)

train_set, _, test_set = get_splits()
test_images, _ = fixed_test_images()
save_shared("test8.pt", {"images": test_images, "labels": []})

def latent_strip(decode, z_a, z_b):
    frames = []
    for alpha in ALPHAS:
        frames.append(decode((1 - alpha) * z_a + alpha * z_b).cpu())
    return torch.cat(frames)


## Loss

Images are scaled to `[-1, 1]` to match the generator's tanh. The discriminator is trained with BCE-with-logits on real and fake batches. The generator is trained with the non-saturating objective: BCE-with-logits of its fakes against the real label. Both Adam learning rates are `2e-4`. A checkpoint is written every epoch so a Colab disconnect can resume.


In [ ]:
generator = Generator().to(DEVICE)
discriminator = Discriminator().to(DEVICE)
generator.apply(dcgan_weights)
discriminator.apply(dcgan_weights)
opt_g = Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion = torch.nn.BCEWithLogitsLoss()

history = {"g_loss": [], "d_loss": [], "epoch_seconds": []}
start_epoch = 0
checkpoint = load_checkpoint(NAME, map_location=DEVICE)
if checkpoint is not None:
    generator.load_state_dict(checkpoint["G"])
    discriminator.load_state_dict(checkpoint["D"])
    opt_g.load_state_dict(checkpoint["opt_G"])
    opt_d.load_state_dict(checkpoint["opt_D"])
    history = load_json(NAME, "history.json")
    start_epoch = checkpoint["epoch"] + 1
    print("resuming at epoch", start_epoch + 1)

for epoch in range(start_epoch, EPOCHS):
    set_seed(SEED + epoch)
    loader = make_loader(train_set, shuffle=True, seed=SEED + epoch)
    generator.train()
    discriminator.train()
    g_total = 0.0
    d_total = 0.0
    seen = 0
    start = time.perf_counter()
    for images, _ in tqdm(loader, leave=False):
        images = to_signed(images.to(DEVICE))
        batch = images.size(0)
        real_labels = torch.ones(batch, device=DEVICE)
        fake_labels = torch.zeros(batch, device=DEVICE)

        noise = torch.randn(batch, NOISE_DIM, device=DEVICE)
        fake = generator(noise).detach()
        d_loss = criterion(discriminator(images), real_labels) + criterion(discriminator(fake), fake_labels)
        opt_d.zero_grad()
        d_loss.backward()
        opt_d.step()

        noise = torch.randn(batch, NOISE_DIM, device=DEVICE)
        g_loss = criterion(discriminator(generator(noise)), real_labels)
        opt_g.zero_grad()
        g_loss.backward()
        opt_g.step()

        g_total += g_loss.item() * batch
        d_total += d_loss.item() * batch
        seen += batch
    elapsed = time.perf_counter() - start
    history["g_loss"].append(g_total / seen)
    history["d_loss"].append(d_total / seen)
    history["epoch_seconds"].append(elapsed)
    save_json(NAME, "history.json", history)
    save_torch(
        NAME,
        "checkpoint.pt",
        {
            "epoch": epoch,
            "G": generator.state_dict(),
            "D": discriminator.state_dict(),
            "opt_G": opt_g.state_dict(),
            "opt_D": opt_d.state_dict(),
        },
    )
    print(f"epoch {epoch + 1:02d}  G {g_total / seen:.3f}  D {d_total / seen:.3f}  {elapsed:.1f}s")


## Samples

The 16-image grid uses noise drawn with seed 69. The interpolation uses two other vectors from seed 70, shared by both GANs. There is no encoder, so MSE, PSNR, and SSIM stay empty. Latency is one generator forward of 64 images.


In [ ]:
checkpoint = load_checkpoint(NAME, map_location=DEVICE)
generator.load_state_dict(checkpoint["G"])
generator.eval()
history = load_json(NAME, "history.json")

with torch.no_grad():
    grid = to_unit(generator(standard_normal(GRID_SAMPLES, NOISE_DIM, SEED).to(DEVICE))).cpu()
    ends = standard_normal(2, NOISE_DIM, SEED + 1).to(DEVICE)
    strip = latent_strip(lambda z: to_unit(generator(z)), ends[0:1], ends[1:2])

set_seed(SEED)
fid_chunks = []
left = FID_SAMPLES
with torch.no_grad():
    while left:
        count = min(128, left)
        noise = torch.randn(count, NOISE_DIM, device=DEVICE)
        fid_chunks.append(to_unit(generator(noise)).cpu())
        left -= count

noise_64 = standard_normal(64, NOISE_DIM, SEED).to(DEVICE)
latency = measure_latency_ms(lambda: generator(noise_64), DEVICE)

save_torch(NAME, "samples_grid.pt", grid)
save_torch(NAME, "interp.pt", strip)
save_torch(NAME, "samples_fid.pt", torch.cat(fid_chunks))
save_json(
    NAME,
    "metrics.json",
    {
        "latent_dim": NOISE_DIM,
        "sec_per_epoch": sum(history["epoch_seconds"]) / len(history["epoch_seconds"]),
        "latency_ms": latency,
        "test_mse": None,
        "test_psnr": None,
        "test_ssim": None,
        "stability": gan_stability(history["g_loss"], history["d_loss"]),
    },
)
show_grid(grid, nrow=4, title="Fixed prior samples")
show_grid(strip, nrow=6, title="Noise interpolation")
print(load_json(NAME, "metrics.json"))
